# Learning Objectives

In this notebook, you will craft sophisticated ETL jobs that interface with a variety of common data sources, such as 
- REST APIs (HTTP endpoints)
- RDBMS
- Hive tables (managed tables)
- Various file formats (csv, json, parquet, etc.)


# Interview Questions

As you progress through the practice, attempt to answer the following questions:

## Columnar File
- What is a columnar file format and what advantages does it offer?

A columnar file format (like Parquet) stores data column-by-column instead of row-by-row, grouping similar values together for each column.

This offers key advantages: it enables better compression (since similar data is stored together) and faster queries, because systems like Spark only read the columns they need instead of scanning entire rows.

- Why is Parquet frequently used with Spark and how does it function?

Apache Parquet is frequently used with Apache Spark because Spark is optimized to take advantage of Parquet’s columnar layout, allowing it to read only the needed columns, compress data efficiently, and skip irrelevant data during queries for much faster performance.

Parquet works by storing data in column chunks grouped into row groups, along with metadata (like min/max values), which Spark uses for optimizations such as column pruning, predicate pushdown, and parallel processing, reducing disk I/O and speeding up large-scale analytics.

- How do you read/write data from/to a Parquet file using a DataFrame?

You read and write Parquet files in PySpark by using the DataFrame API—spark.read.parquet("path") loads Parquet data into a DataFrame, and df.write.parquet("path") saves the DataFrame back to disk in efficient columnar Parquet format.


## Partitions
- How do you save data to a file system by partitions? (Hint: Provide the code)

You save data by partitions by using df.write.partitionBy("column").saveAsTable(...), which organizes the output into separate directory folders based on unique values of that column, improving query performance by enabling partition pruning.

- How and why can partitions reduce query execution time? (Hint: Give an example)

Partitions reduce query execution time by organizing data into separate directories based on column values, so Spark can skip irrelevant partitions instead of scanning the entire dataset. For example, if a sales dataset is partitioned by year, a query for only 2023 sales will read just the year=2023 partition rather than all years, significantly reducing I/O and processing time. This technique, called partition pruning, improves performance for large datasets.

## JDBC and RDBMS
- How do you load data from an RDBMS into Spark? (Hint: Discuss the steps and JDBC)

To load data from a relational database into Spark, you use Spark’s JDBC connector by specifying the database URL, table or query, and authentication credentials. Spark reads the data in parallel by optionally providing a partitioning column, lower/upper bounds, and number of partitions to optimize distributed loading. The typical steps are: establish the JDBC connection, read the table or query into a DataFrame using spark.read.format("jdbc").option(...).load(), and then perform Spark transformations or write the data to a target storage.

## REST API and HTTP Requests
- How can Spark be used to fetch data from a REST API? (Hint: Discuss making API requests)

Spark can fetch data from a REST API by using Python or Scala code on the driver to make HTTP requests (e.g., via requests in Python) and then converting the returned JSON or CSV data into a Spark DataFrame. Typically, you loop over API endpoints or parameters, fetch data, clean/transform it locally, and use spark.createDataFrame() to parallelize it for distributed processing. Care must be taken to respect API rate limits by adding delays or retries to avoid throttling or errors.

## ETL Job One: Parquet file
### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Data transformation requirements https://pgexercises.com/questions/aggregates/fachoursbymonth.html

### Load
Load data into a parquet file

### What is Parquet? 

Columnar files are an important technique for optimizing Spark queries. Additionally, they are often tested in interviews.
- https://www.youtube.com/watch?v=KLFadWdomyI
- https://www.databricks.com/glossary/what-is-parquet

In [0]:
from pyspark.sql.functions import col, sum

# Extract
df_bookings = spark.sql("select * from bookings")

# Transform
df_fac_hours_month = df_bookings \
                        .where(col("starttime").startswith("2012-09")) \
                        .groupBy("facid") \
                        .agg(sum("slots").alias("Total Slots")) \
                        .select("facid",  "Total Slots") \
                        .orderBy("Total Slots")

# Load
file_location = "/Volumes/jrvs_databricks_workspace/default/filestorage/fac_hours_2012-09"
df_fac_hours_month.write.mode('overwrite').parquet(file_location)


## ETL Job Two: Partitions

### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Transform the data https://pgexercises.com/questions/joins/threejoin.html

### Load
Partition the result data by facility column and then save to `threejoin_delta` managed table. Additionally, they are often tested in interviews.

hint: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.partitionBy.html

What are paritions? 

Partitions are an important technique to optimize Spark queries
- https://www.youtube.com/watch?v=hvF7tY2-L3U&t=268s

In [0]:
# Write your solution here
from pyspark.sql.functions import concat, lit

# Extract
df_facilities = spark.sql("select * from facilities")
df_bookings = spark.sql("select * from bookings")
df_members = spark.sql("select * from members")

# Transform
df_members_tennis = df_bookings \
                                    .join(df_facilities, "facid") \
                                    .join(df_members, "memid") \
                                    .withColumn("member", concat(col("firstname"), lit(" ") ,col("surname"))) \
                                    .select("member", col("name").alias("facility")) \
                                    .dropDuplicates() \
                                    .filter(col("facility").startswith("Tennis")) \
                                    .orderBy(["member", "facility"])

df_members_tennis.write.mode('overwrite').partitionBy("facility").saveAsTable("threejoin_delta")

## ETL Job Three: HTTP Requests

### Extract
Extract daily stock price data price from the following companies, Google, Apple, Microsoft, and Tesla. 

Data Source
- API: https://rapidapi.com/alphavantage/api/alpha-vantage
- Endpoint: GET `TIME_SERIES_DAILY`

Sample HTTP request

```
curl --request GET \
	--url 'https://alpha-vantage.p.rapidapi.com/query?function=TIME_SERIES_DAILY&symbol=TSLA&outputsize=compact&datatype=json' \
	--header 'X-RapidAPI-Host: alpha-vantage.p.rapidapi.com' \
	--header 'X-RapidAPI-Key: [YOUR_KEY]'

```

Sample Python HTTP request

```
import requests

url = "https://alpha-vantage.p.rapidapi.com/query"

querystring = {
    "function":"TIME_SERIES_DAILY",
    "symbol":"IBM",
    "datatype":"json",
    "outputsize":"compact"
}

headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": "[YOUR_KEY]"
}

response = requests.get(url, headers=headers, params=querystring)

data = response.json()

# Now 'data' contains the daily time series data for "IBM"
```

### Transform
Find **weekly** max closing price for each company.

hints: 
  - Use a `for-loop` to get stock data for each company
  - Use the spark `union` operation to concat all data into one DF
  - create a new `week` column from the data column
  - use `group by` to calcualte max closing price

### Load
- Partition `DF` by company
- Load the DF in to a managed table called, `max_closing_price_weekly`

In [0]:
# Extract

# Sample request
# curl --request GET --url 'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol=TSLA&apikey=MWYLSDGZ1EJU503D'

import requests
import time
from functools import reduce
from pyspark.sql.functions import lit

API_KEY = "MWYLSDGZ1EJU503D"
url = "https://www.alphavantage.co/query"

def fetch_stock_data(symbol):
    querystring = {
        "function":"TIME_SERIES_DAILY",
        "symbol":symbol,
        "datatype":"json",
        "outputsize":"compact",
        "apikey" : API_KEY
    }
    response = requests.get(url, params=querystring)
    return response.json()

def clean_stock_data(json_string):
    ts = json_string["Time Series (Daily)"]
    pdf = pd.DataFrame.from_dict(ts, orient="index")
    pdf.reset_index(inplace=True)
    pdf.rename(columns={"index": "date"}, inplace=True)
    pdf.columns = [
        "date",
        "open",
        "high",
        "low",
        "close",
        "volume"
    ]
    return pdf

companies = ["IBM", "AAPL", "MSFT", "TSLA", "GOOG"]

dfs = []

for company in companies:
    json_data = fetch_stock_data(company)
    pdf = clean_stock_data(json_data)

    df = spark.createDataFrame(pdf)

    # Transform
    df = df.withColumn("company", lit(company))

    dfs.append(df)
    time.sleep(1.3)

df_time_series = reduce(lambda df1, df2: df1.union(df2), dfs)

# Load
df_time_series.write.mode('overwrite').partitionBy("company").saveAsTable("max_closing_price_weekly")

## ETL Job Four: RDBMS


### Extract
Extract RNA data from a public PostgreSQL database.

- https://rnacentral.org/help/public-database
- Extract 100 RNA records from the `rna` table (hint: use `limit` in your sql)
- hint: use `spark.read.jdbc` https://docs.databricks.com/external-data/jdbc.html

### Transform
We want to load the data as it so there is no transformation required.


### Load
Load the DF in to a managed table called, `rna_100_records`

In [0]:
%sh nc -vz hh-pgsql-public.ebi.ac.uk 5432

hh-pgsql-public.ebi.ac.uk [193.62.192.243] 5432 (postgresql) open


In [0]:
# Extract
rna_table = (spark.read
  .format("jdbc")
  .option("url", "jdbc:postgresql://hh-pgsql-public.ebi.ac.uk:5432/pfmegrnargs")
  .option("dbtable", "(SELECT * FROM rna LIMIT 100) as tmp")
  .option("driver", "org.postgresql.Driver")
  .option("user", "reader")
  .option("password", "NWDMCE5xdipIjRrp")
  .load()
)

# Transform

# Load
rna_table.write.mode("overwrite").saveAsTable("jrvs_databricks_workspace.default.rna_100_records")